# Session 2 — Broadcasting, matmul, einsum, vectorization

Companion to [../numpy_pytorch_schedule.md](../numpy_pytorch_schedule.md). Where NumPy stops being "loops with extra steps." Every idea here transfers verbatim to PyTorch.

In [4]:
import numpy as np

## Broadcasting — the superpower and the footgun

Arrays of different shapes combine element-wise by stretching size-1 (or missing) dims. Rules, applied **right-to-left**:
1. Pad the shorter shape with 1s on the **left**.
2. Each axis: sizes must be **equal**, or one must be **1** (the 1 stretches).
3. Otherwise: error.

In [5]:
a = np.ones((3, 4))
print((a + np.ones(4)).shape)        # (4,) -> (1,4) -> (3,4)   OK
print((a + np.ones((3, 1))).shape)   # stretch over columns     OK
try:
    a + np.ones(3)                   # aligns 4 vs 3 -> error
except ValueError as e:
    print("error:", e)

(3, 4)
(3, 4)
error: operands could not be broadcast together with shapes (3,4) (3,) 


Two shapes that **are** Transformer code — a per-feature gain over `(B,S,D)`, and an outer-product-style `(B,S,1)*(B,1,S) -> (B,S,S)`:

In [6]:
x = np.random.randn(2, 5, 8)          # (B,S,D)
gain = np.random.randn(8)             # (D,)
print("(B,S,D)*(D,):", (x * gain).shape)         # broadcasts over B,S

u = np.random.randn(2, 5, 1)
v = np.random.randn(2, 1, 5)
print("(B,S,1)*(B,1,S):", (u * v).shape)         # (2,5,5)

(B,S,D)*(D,): (2, 5, 8)
(B,S,1)*(B,1,S): (2, 5, 5)


> **Footgun:** `(4,1)` and `(1,4)` silently broadcast to `(4,4)` when you meant a `(4,)` result. A suspiciously square/large output is usually an accidental broadcast.

## Matrix multiply and batched matmul

`@` (or `np.matmul`) multiplies the **last two axes**; all **leading axes broadcast** as batch. One batched matmul computes an `S×S` score matrix for every (batch, head) at once — the whole "attention is GPU-efficient" story.

In [7]:
A = np.random.randn(3, 4); B = np.random.randn(4, 5)
print("A@B:", (A @ B).shape)          # (3,5)

Q = np.random.randn(2, 4, 6, 8)       # (B,H,S,D_h)
K = np.random.randn(2, 4, 6, 8)
scores = Q @ K.transpose(0, 1, 3, 2)  # transpose last two -> (2,4,8,6)
print("scores (B,H,S,S):", scores.shape)   # (2,4,6,6)

A@B: (3, 5)
scores (B,H,S,S): (2, 4, 6, 6)


## einsum — say the shape you want

`np.einsum` names each axis and states the output. Repeated index = multiply-and-sum over it; an index missing from the output = summed out. Being able to read `einsum('bsd,btd->bst', q, k)` as "attention scores" is a fluency milestone.

In [ ]:
Q = np.random.randn(2, 6, 8); K = np.random.randn(2, 6, 8)
scores = np.einsum('bsd,btd->bst', Q, K)          # out[b,s,t] = sum_d Q[b,s,d]*K[b,t,d]
print("einsum scores:", scores.shape)             # (2,6,6)
# einsum == batched matmul:
print("matches matmul:", np.allclose(scores, Q @ K.transpose(0, 2, 1)))

A = np.arange(6).reshape(2, 3)
print("transpose:", np.einsum('ij->ji', A).shape) # (3,2)
print("row sums :", np.einsum('ij->i', A))        # sum out j
print("outer    :", np.einsum('i,j->ij', np.arange(3), np.arange(4)).shape)  # (3,4)

## Vectorization mindset

Replace element loops with whole-array ops — faster (runs in C) and idiomatic. If you're looping over rows/tokens, ask whether broadcasting, matmul, or einsum does it in one expression.

In [ ]:
x = np.random.randn(6)
# slow: for i in range(len(x)): out[i] = max(0, x[i])
out = np.maximum(0, x)                # ReLU, no loop
print(out)

## Self-check

1. Broadcast & result shape: `(5,3)+(3,)`, `(5,3)+(5,)`, `(5,1)+(1,3)`, `(2,3,4)+(4,)`.
2. For `Q,K` of `(B,H,S,D_h)`, write the matmul giving scores `(B,H,S,S)`. Which axes are batch vs matrix?
3. What does `einsum('bhsd,bhtd->bhst', Q, K)` compute?

**Answers.** (1) `(5,3)` ✓; **error** (5 vs 3); `(5,3)` ✓; `(2,3,4)` ✓. (2) `Q @ K.transpose(0,1,3,2)` — `B,H` batch, last two `S,D_h`×`D_h,S` are the matrices. (3) `scores[b,h,s,t]=Σ_d Q[b,h,s,d]·K[b,h,t,d]` — identical to (2).

## Exercise — softmax + attention in raw NumPy

Implement a numerically-stable softmax over the last axis, then `out = softmax(QKᵀ/√d) @ V` for `Q,K,V` of shape `(S,d)`. Confirm each row of the weights sums to 1.

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)      # stability: subtract the max
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)   # keepdims so it broadcasts back

def attention(Q, K, V):
    d = Q.shape[-1]
    A = softmax((Q @ K.T) / np.sqrt(d), axis=-1) # softmax over the KEY axis
    return A @ V, A

S, d = 4, 8
Q, K, V = (np.random.randn(S, d) for _ in range(3))
out, A = attention(Q, K, V)
print("out shape:", out.shape)          # (4,8)
print("rows sum to 1:", np.round(A.sum(axis=-1), 6))   # ~[1 1 1 1]

You've written scaled dot-product attention in raw NumPy. The `keepdims` softmax, the `/√d` scale, and "softmax over the key axis" are exactly what PyTorch does — Session 3 ports this to differentiable tensors.